# DizerCore LoRA Training

Trains a TrinityCore investigation-prompt LoRA adapter. Auto-detects
single vs dual T4 and uses DDP when 2 GPUs are present.

**Setup:**

1. Upload `dizercore-dataset.jsonl` as a Kaggle dataset (from the Pi's
   Training tab). Note its path.
2. **Settings → Accelerator → T4 x2**.
3. Set `DATASET` in Cell 2 to the path shown in the Data panel.
4. **Save Version → Save & Run All (Commit)**.

**Runtime:** ~6.5 h on 2×T4 (DDP), ~11 h on 1×T4. The 2-GPU path fits
comfortably inside the 12 h commit limit. The 1-GPU path is tight —
save the output as a Kaggle dataset mid-run if possible.

**Output:** `/kaggle/working/dizercore/dizercore-q4_k_m.gguf` — download
from the Output panel, then upload via the Pi's Web UI Training tab.

**Base model:** read from the dataset meta row (line 1) so the adapter
matches whatever model the Pi has installed.

**Sequence packing:** the whole token stream is concatenated and sliced
into fixed 2048-token chunks — nothing is truncated, nothing is padded.

In [ ]:
# ============ CELL 1 — Install ============
import subprocess, sys, os
os.environ['PYTHONUNBUFFERED'] = '1'

print('=== Install ===', flush=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade',
                'transformers==4.46.3', 'peft==0.13.2',
                'accelerate==1.1.1', 'datasets==3.1.0'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade',
                'bitsandbytes'], check=True)

print('=== GPUs visible ===', flush=True)
print(subprocess.run(['nvidia-smi', '-L'], capture_output=True, text=True).stdout)

In [ ]:
# ============ CELL 2 — Write the training script ============
%%writefile /kaggle/working/train_ddp.py
import os
# CRITICAL: set BEFORE any torch import
os.environ['NCCL_P2P_DISABLE'] = '1'
os.environ['NCCL_SHM_DISABLE'] = '1'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
os.environ['HF_XET_HIGH_PERFORMANCE'] = '1'

import gc, json, shutil, glob, subprocess, sys
import torch
import torch.distributed as dist
from transformers import (AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig,
                          TrainingArguments, Trainer,
                          DataCollatorForLanguageModeling)
from peft import LoraConfig, get_peft_model, PeftModel
from datasets import load_dataset

local_rank = int(os.environ.get('LOCAL_RANK', 0))
world_size = int(os.environ.get('WORLD_SIZE', 1))
is_main = (local_rank == 0)
torch.cuda.set_device(local_rank)

dist.init_process_group(backend='nccl')

def p(*a, **kw):
    if is_main:
        print(*a, **kw, flush=True)

p(f'=== DDP worker {local_rank}/{world_size} on GPU {torch.cuda.current_device()} ===')

# ============ CONFIG ============
DATASET = '/kaggle/input/datasets/vekzla/dizercore-wow/dizercore-dataset.jsonl'
WORK = '/kaggle/working/dizercore'
ADAPTER = f'{WORK}/adapter'
MERGED = f'{WORK}/merged'
GGUF_F16 = f'{WORK}/dizercore-f16.gguf'
GGUF_Q4 = f'{WORK}/dizercore-q4_k_m.gguf'
EPOCHS = 1
LR = 2e-4
MAX_LEN = 2048
MICRO_BATCH = 2
GRAD_ACCUM = 8
WARMUP_STEPS = 10

if is_main:
    os.makedirs(WORK, exist_ok=True)

# ---------- auto-detect checkpoints from attached datasets ----------
PREV_CHECKPOINTS = None
for pattern in ('/kaggle/input/*/checkpoints',
                '/kaggle/input/*/*/checkpoints',
                '/kaggle/input/*/dizercore/checkpoints'):
    hits = sorted(glob.glob(pattern))
    if hits:
        best = max(hits, key=lambda p: max(
            (int(d.rsplit('-', 1)[-1]) for d in os.listdir(p)
             if d.startswith('checkpoint-') and d.rsplit('-', 1)[-1].isdigit()),
            default=-1))
        PREV_CHECKPOINTS = best
        p(f'Auto-detected checkpoints: {PREV_CHECKPOINTS}')
        break
if PREV_CHECKPOINTS is None:
    p('No checkpoint dataset attached — training from scratch')

BASE = 'Qwen/Qwen2.5-3B-Instruct'
try:
    with open(DATASET) as _f:
        BASE = json.loads(_f.readline()).get('base_model', BASE)
except Exception as _e:
    p(f'No base_model tag ({_e}); using default')
p('Base model:', BASE)
p('Dataset:', DATASET)

# ============ DATASET ============
p('=== Load dataset ===')
ds = load_dataset('json', data_files=DATASET, split='train')
ds = ds.filter(lambda x: not x.get('_meta', False))
p(f'{len(ds)} training examples')

SYSTEM_PROMPT = ('You are a software engineer building a TrinityCore WoW '
                 'emulation server. You investigate issues and produce '
                 'structured prompts listing which files/folders to check and '
                 'what to verify - SQL or C++ code. You never write the fix '
                 'itself.')

def fmt(ex):
    user = ex['instruction']
    if ex.get('input'):
        user += '\n\n' + ex['input']
    return {'text': (f'<|im_start|>system\n{SYSTEM_PROMPT}<|im_end|>\n'
                     f'<|im_start|>user\n{user}<|im_end|>\n'
                     f'<|im_start|>assistant\n{ex["output"]}<|im_end|>')}

ds = ds.map(fmt, remove_columns=[c for c in ds.column_names if c != 'text'])

# ============ TOKENIZER ============
tokenizer = AutoTokenizer.from_pretrained(BASE, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
EOS = tokenizer.eos_token_id

# ============ PACK ============
p('=== Tokenizing + packing ===')
ds = ds.shuffle(seed=42)

def pack_batch(batch):
    stream = []
    for t in batch['text']:
        stream.extend(tokenizer(t, add_special_tokens=False)['input_ids'])
    chunks = []
    for i in range(0, len(stream) - MAX_LEN + 1, MAX_LEN):
        chunks.append(stream[i:i + MAX_LEN])
    rem = len(stream) % MAX_LEN
    if rem >= MAX_LEN // 4:
        chunks.append(stream[-rem:] + [EOS] * (MAX_LEN - rem))
    return {
        'input_ids': chunks,
        'attention_mask': [[1] * MAX_LEN for _ in chunks],
        'length': [MAX_LEN] * len(chunks),
    }

tok = ds.map(pack_batch, batched=True, batch_size=2000,
             remove_columns=['text'], num_proc=1)
p(f'Packed into {len(tok)} sequences of {MAX_LEN} tokens')
del ds; gc.collect()

# ============ BASE MODEL ============
p('=== Load base model (4-bit) ===')
bnb = BitsAndBytesConfig(load_in_4bit=True,
                         bnb_4bit_quant_type='nf4',
                         bnb_4bit_compute_dtype=torch.float16,
                         bnb_4bit_use_double_quant=True)
model = AutoModelForCausalLM.from_pretrained(
    BASE,
    quantization_config=bnb,
    device_map={'': local_rank},
    attn_implementation='sdpa',
)
model.config.use_cache = False
model.gradient_checkpointing_enable(
    gradient_checkpointing_kwargs={'use_reentrant': False})

# ============ LoRA ============
lora = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, bias='none',
                  task_type='CAUSAL_LM',
                  target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj',
                                  'gate_proj', 'up_proj', 'down_proj'])
model = get_peft_model(model, lora)
if is_main:
    model.print_trainable_parameters()

# ============ CHECKPOINT RESTORE ============
ckpt_dir = os.path.join(WORK, 'checkpoints')
if is_main and PREV_CHECKPOINTS and os.path.isdir(PREV_CHECKPOINTS):
    p(f'Restoring from {PREV_CHECKPOINTS}')
    os.makedirs(ckpt_dir, exist_ok=True)
    for c in os.listdir(PREV_CHECKPOINTS):
        src = os.path.join(PREV_CHECKPOINTS, c)
        dst = os.path.join(ckpt_dir, c)
        if os.path.isdir(src) and not os.path.exists(dst):
            shutil.copytree(src, dst)
dist.barrier()

ckpt = None
if os.path.isdir(ckpt_dir):
    cps = [os.path.join(ckpt_dir, d) for d in os.listdir(ckpt_dir)
           if d.startswith('checkpoint-') and d.rsplit('-', 1)[-1].isdigit()]
    if cps:
        ckpt = max(cps, key=lambda p: int(p.rsplit('-', 1)[-1]))
        p(f'Will resume from {ckpt}')

# ============ TRAIN ============
args = TrainingArguments(
    output_dir=ckpt_dir,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=MICRO_BATCH,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LR,
    lr_scheduler_type='cosine',
    warmup_steps=WARMUP_STEPS,
    logging_steps=1,
    save_steps=25,
    save_total_limit=4,
    fp16=True,
    bf16=False,
    optim='paged_adamw_8bit',
    report_to='none',
    disable_tqdm=(not is_main),
    dataloader_num_workers=1,
    dataloader_pin_memory=True,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={'use_reentrant': False},
    ddp_find_unused_parameters=False,
    ddp_backend='nccl',
)

trainer = Trainer(model=model, args=args, train_dataset=tok,
                  data_collator=DataCollatorForLanguageModeling(tokenizer, mlm=False))

p('=== Train ===')
trainer.train(resume_from_checkpoint=ckpt)

# ============ SAVE ADAPTER ============
dist.barrier()
if is_main:
    p('=== Saving adapter ===')
    os.makedirs(ADAPTER, exist_ok=True)
    model.save_pretrained(ADAPTER)
    tokenizer.save_pretrained(ADAPTER)
del model; gc.collect(); torch.cuda.empty_cache()

dist.barrier()
dist.destroy_process_group()

# ============ MERGE + GGUF — rank 0 only ============
if not is_main:
    sys.exit(0)

p('=== Merging adapter into base ===')
base = AutoModelForCausalLM.from_pretrained(BASE, torch_dtype=torch.float16,
                                            device_map={'': 0})
merged = PeftModel.from_pretrained(base, ADAPTER).merge_and_unload()
os.makedirs(MERGED, exist_ok=True)
merged.save_pretrained(MERGED)
tokenizer.save_pretrained(MERGED)
del base, merged; gc.collect(); torch.cuda.empty_cache()

p('=== Building llama.cpp ===')
LLAMA_DIR = f'{WORK}/llama.cpp'
if not os.path.isdir(LLAMA_DIR):
    subprocess.run(['git', 'clone', '--depth', '1',
                    'https://github.com/ggerganov/llama.cpp', LLAMA_DIR], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r',
                f'{LLAMA_DIR}/requirements.txt'], check=True)
subprocess.run(['cmake', '-B', f'{LLAMA_DIR}/build', '-DGGML_NATIVE=ON',
                '-DLLAMA_CURL=OFF'], cwd=LLAMA_DIR, check=True)
subprocess.run(['cmake', '--build', f'{LLAMA_DIR}/build', '--config', 'Release',
                '-j', '4'], cwd=LLAMA_DIR, check=True)

p('=== Converting to GGUF ===')
subprocess.run([sys.executable, f'{LLAMA_DIR}/convert_hf_to_gguf.py', MERGED,
                '--outfile', GGUF_F16, '--outtype', 'f16'], check=True)

p('=== Quantizing to Q4_K_M ===')
subprocess.run([f'{LLAMA_DIR}/build/bin/llama-quantize',
                GGUF_F16, GGUF_Q4, 'Q4_K_M'], check=True)

size_mb = os.path.getsize(GGUF_Q4) / 1024 / 1024
p('=== DONE ===')
p(f'Base trained: {BASE}')
p(f'GGUF: {GGUF_Q4} ({size_mb:.0f} MB)')
p('Download from Output panel: /kaggle/working/dizercore/dizercore-q4_k_m.gguf')

In [ ]:
# ============ CELL 3 — Launch (auto-selects 1 or 2 GPUs) ============
import subprocess, os

n_gpus = int(subprocess.run(
    ['python', '-c', 'import torch; print(torch.cuda.device_count())'],
    capture_output=True, text=True).stdout.strip())

print(f'Detected {n_gpus} GPU(s)', flush=True)

if n_gpus >= 2:
    print('=== Launching DDP with 2 GPUs ===', flush=True)
    result = subprocess.run([
        'torchrun', '--standalone',
        '--nproc_per_node=2',
        '--master_port=29501',
        '/kaggle/working/train_ddp.py',
    ], text=True)
else:
    print('=== Launching single-GPU ===', flush=True)
    env = {**os.environ,
           'CUDA_VISIBLE_DEVICES': '0',
           'RANK': '0', 'LOCAL_RANK': '0',
           'WORLD_SIZE': '1', 'MASTER_ADDR': '127.0.0.1',
           'MASTER_PORT': '29501'}
    result = subprocess.run(['python', '/kaggle/working/train_ddp.py'],
                            env=env, text=True)

print(f'\n=== Exit code: {result.returncode} ===', flush=True)

## Resuming from a checkpoint

If a previous run died mid-training, save `/kaggle/working/dizercore/checkpoints`
as a Kaggle dataset. In the next session, attach it as Input. Cell 2
auto-detects it under `/kaggle/input/*/checkpoints` and resumes from the
highest-numbered checkpoint.

No manual `PREV_CHECKPOINTS` edit needed.

## Single-GPU fallback

Cell 3 auto-detects GPU count. With 1 GPU it runs the same script as a
plain process (no DDP), taking ~11 h. With 2 GPUs it uses `torchrun`,
taking ~6.5 h. Same code path, same output.

## Downloading the GGUF

After the commit finishes, on the notebook version page:

1. Right sidebar → **Output** section
2. Expand `/kaggle/working` → `dizercore/`
3. Click `dizercore-q4_k_m.gguf` → **Download**

To make it permanent, right-click `dizercore/` → **Create Dataset**.
Then you can pull it on the Pi with:

```bash
kaggle datasets download <your-user>/dizercore-checkpoints
```